<a href="https://colab.research.google.com/github/toche7/AI_ITM/blob/main/linear_regression_special.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Special Session: Linear Regression
**Python + scikit-learn (English)**
Duration: 1 hour 30 minutes
Goal: Build, evaluate, and interpret regression models with confidence.


## Session Plan (90 minutes)
1. Problem framing and regression intuition (10 min)
2. Math behind linear regression (10 min)
3. End-to-end sklearn workflow (25 min)
4. Evaluation and diagnostics (20 min)
5. Improving models: features + regularization (15 min)
6. Guided practice and recap (10 min)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer


## Part A: Simple Dataset First (cost.xlsx)

Use case: airplane fuel cost per trip vs number of passengers.


In [ ]:
# Load simple Excel dataset from GitHub

simple_url = "https://raw.githubusercontent.com/toche7/DataSets/main/cost.xlsx"



df = pd.read_excel(simple_url)

df["time"] = pd.to_datetime(df["time"])

target_col = "Fuel Cost"



print("Simple dataset shape:", df.shape)

print("Columns:", list(df.columns))

print("Selected target:", target_col)

display(df.head())


In [ ]:
df.describe().T


In [ ]:
# Visualize relationship between passengers and fuel cost

plt.figure(figsize=(7,4))

sns.scatterplot(data=df, x="Passenger", y=target_col)

plt.title("Fuel Cost vs Passenger")

plt.show()


## Step 1: Define X and y (Baseline)

For the baseline model, we use only `Passenger` as the feature.


In [ ]:
X = df[["Passenger"]]

y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Features:", list(X.columns))

print("X_train:", X_train.shape, "X_test:", X_test.shape)


## Baseline Linear Regression


In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)
print("Intercept:", lr.intercept_)
print("MAE:", round(mae, 4))
print("RMSE:", round(rmse, 4))
print("R2:", round(r2, 4))


In [ ]:
pd.DataFrame({"actual": y_test.values[:10], "predicted": y_pred[:10]})


## Coefficients


In [ ]:
coef_df = pd.DataFrame({"feature": X.columns, "coefficient": lr.coef_}).sort_values("coefficient", ascending=False)
coef_df


## Diagnostics


In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred, alpha=0.35)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs Predicted")
plt.show()


In [ ]:
residuals = y_test - y_pred
plt.figure(figsize=(7,4))
plt.scatter(y_pred, residuals, alpha=0.35)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Residual Plot")
plt.show()


## Pipeline, Cross-Validation, and Regularization


In [ ]:
pipe = Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())])
pipe.fit(X_train, y_train)
y_pred_pipe = pipe.predict(X_test)
print("Pipeline RMSE:", round(mean_squared_error(y_test, y_pred_pipe) ** 0.5, 4))
print("Pipeline R2:", round(r2_score(y_test, y_pred_pipe), 4))
scores = cross_val_score(LinearRegression(), X, y, cv=5, scoring="neg_root_mean_squared_error")
print("CV RMSE mean:", round((-scores).mean(), 4))


In [ ]:
models = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(alpha=1.0, random_state=42),
    "Lasso": Lasso(alpha=0.01, random_state=42)
}
rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rows.append({"model": name, "RMSE": mean_squared_error(y_test, pred) ** 0.5, "R2": r2_score(y_test, pred)})
pd.DataFrame(rows).sort_values("RMSE")


## Part B: California Housing (Step-by-Step)

Now we repeat the same learning pattern from Part A, one step at a time.


In [ ]:
# Step B1: Load and inspect dataset

housing = fetch_california_housing(as_frame=True)

df_ca = housing.frame.copy()

target_ca = "MedHouseVal"



print("California Housing shape:", df_ca.shape)

print("Target:", target_ca)

display(df_ca.head())


## Step B2: Define features and target

Use all available predictors except the target column.


In [ ]:
X_ca = df_ca.drop(columns=[target_ca])

y_ca = df_ca[target_ca]



print("Number of features:", X_ca.shape[1])

print("X_ca shape:", X_ca.shape)

print("y_ca shape:", y_ca.shape)


## Step B3: Train/test split

Keep the same split setup so results are reproducible.


In [ ]:
X_train_ca, X_test_ca, y_train_ca, y_test_ca = train_test_split(

    X_ca, y_ca, test_size=0.2, random_state=42

)



print("X_train_ca:", X_train_ca.shape)

print("X_test_ca:", X_test_ca.shape)


## Step B4: Train baseline linear regression and evaluate

Compute MAE, RMSE, and R2 exactly like Part A.


In [ ]:
lr_ca = LinearRegression()

lr_ca.fit(X_train_ca, y_train_ca)

y_pred_ca = lr_ca.predict(X_test_ca)



mae_ca = mean_absolute_error(y_test_ca, y_pred_ca)

rmse_ca = mean_squared_error(y_test_ca, y_pred_ca) ** 0.5

r2_ca = r2_score(y_test_ca, y_pred_ca)



print("Intercept:", lr_ca.intercept_)

print("MAE:", round(mae_ca, 4))

print("RMSE:", round(rmse_ca, 4))

print("R2:", round(r2_ca, 4))


## Step B5: Inspect coefficients

Review feature effects for the California dataset.


In [ ]:
coef_ca = pd.DataFrame({

    "feature": X_ca.columns,

    "coefficient": lr_ca.coef_

}).sort_values("coefficient", ascending=False)



display(coef_ca.head(10))


## In-Class Exercise 1
1. Train a baseline LinearRegression model.
2. Compute MAE, RMSE, and R2.
3. Print top 5 positive and negative coefficients.
4. Write one paragraph interpreting the output.


In [ ]:
# Student workspace for Exercise 1


## In-Class Exercise 2
1. Add scaling with a Pipeline.
2. Train Ridge and Lasso.
3. Compare RMSE and R2 with baseline.
4. Decide which model you would deploy first and explain why.


In [ ]:
# Student workspace for Exercise 2


## Recap
- Linear regression is a strong, interpretable baseline.
- Metrics and diagnostics are both necessary.
- Pipelines help prevent leakage and improve reproducibility.
- Regularization helps with stability and generalization.
- Iterative workflow beats one-shot modeling.
